In [1]:
import torch
from dataset_utils import *
import numpy as np
import umap
from pipeline_utils import create_profile_vector, create_tweet_vectors
from sklearn.preprocessing import RobustScaler
from transformers import DistilBertTokenizer, AutoModel
import joblib

/home/max/ProgrammingProjects/-Social-Media-Bot-Detection-with-Continuous-Learning/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def create_raw_embedding(data_loader, device = "cuda"):
    # -- initial setup
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
    scaler = RobustScaler(with_centering=False)

    # -- training loop
    ground_truth_labels = []
    profile_vectors = []
    tweet_vectors = []
    for i, sample in enumerate(data_loader):
        # get feature vectors
        profile_embed = create_profile_vector(sample.user_data)
        tweet_embeds = create_tweet_vectors(sample.tweet_data, tokenizer, model, max_tweets= 300, batch_size = 300)
        # pool tweet vectors
        tweet_vec = torch.mean(tweet_embeds, dim=0, dtype=torch.float32)

        # append feature vectors and ground truth
        profile_vectors.append(profile_embed)
        tweet_vectors.append(tweet_vec)
        ground_truth_labels.append(sample.label)
        print(f"\rIteration: {i} | {sample.label}", end="")
    # scale profile vectors
    scaled_profile_vectors = scaler.fit_transform(profile_vectors)


    return tweet_vectors, scaled_profile_vectors, ground_truth_labels

def save_processed_dataset(train_dataset, test_dataset, dataset_name, umap_reducer, train_umap = False, root_path = "./datasets/ProcessedDatasets/", device="cuda"):
    # process training dataset
    tweet_vectors, profile_vectors, labels = create_raw_embedding(train_dataset, device = "cuda")
    if train_umap: train_reduced_tweet_vectors = umap_reducer.fit_transform(X=tweet_vectors, y=torch.tensor(labels))
    else: train_reduced_tweet_vectors = umap_reducer.transform(tweet_vectors)
    train_reduced_tweet_vectors = np.nan_to_num(train_reduced_tweet_vectors, nan=0.0, posinf=0.0, neginf=0.0)
    train_embedding_features = [np.concatenate((t1, t2), axis=0) for t1, t2 in zip(train_reduced_tweet_vectors, profile_vectors)]

    # safe values
    embeddings = torch.tensor(np.array(train_embedding_features))
    ground_truths = torch.tensor(np.array(labels))
    embeddings = embeddings.cpu().detach()
    ground_truths = ground_truths.cpu().detach()

    torch.save(embeddings, f"{root_path}{dataset_name}TrainEmbed.pt")
    torch.save(ground_truths, f"{root_path}{dataset_name}TrainLabel.pt")

    # process test dataset
    tweet_vectors, profile_vectors, labels = create_raw_embedding(test_dataset, device = "cuda")
    reduced_tweet_vectors = umap_reducer.transform(tweet_vectors)
    reduced_tweet_vectors = np.nan_to_num(reduced_tweet_vectors, nan=0.0, posinf=0.0, neginf=0.0)
    embedding_features = [np.concatenate((t1, t2), axis=0) for t1, t2 in zip(reduced_tweet_vectors, profile_vectors)]

    # safe values
    embeddings = torch.tensor(np.array(embedding_features))
    ground_truths = torch.tensor(np.array(labels))
    embeddings = embeddings.cpu().detach()
    ground_truths = ground_truths.cpu().detach()

    torch.save(embeddings, f"{root_path}{dataset_name}TestEmbed.pt")
    torch.save(ground_truths, f"{root_path}{dataset_name}TestLabel.pt")

In [3]:
# Dataset: Cresci17
dim_reducer = umap.UMAP(n_components=38, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)

train_dataset = InterleavedIterableDataset([
            Cresci17(Cresci17SetTypes.GENUINE_USER, "train", root="./datasets", custom_label=0),
            Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, "train", root="./datasets", custom_label=1),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, "train", root="./datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, "train", root="./datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, "train", root="./datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, "train", root="./datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_2, "train", root="./datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_3, "train", root="./datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_4, "train", root="./datasets", custom_label=3),
        ], "Random")

test_dataset = InterleavedIterableDataset([
            Cresci17(Cresci17SetTypes.GENUINE_USER, "test", root="./datasets", custom_label=0),
            Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, "test", root="./datasets", custom_label=1),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, "test", root="./datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, "test", root="./datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, "test", root="./datasets", custom_label=2),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, "test", root="./datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_2, "test", root="./datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_3, "test", root="./datasets", custom_label=3),
            Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_4, "test", root="./datasets", custom_label=3),
        ], "Random")

save_processed_dataset(train_dataset, test_dataset, "Cresci17", dim_reducer, True, "./datasets/ProcessedDatasets/", device="cuda")
joblib.dump(dim_reducer, "./datasets/ProcessedDatasets/UmapModel.joblib")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9881.27it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 10144 | 2

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9397.32it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 1310 | 2

['./datasets/ProcessedDatasets/UmapModel.joblib']

In [3]:
# Dataset: Cresci18
dim_reducer = joblib.load("./datasets/ProcessedDatasets/UmapModel.joblib")

train_dataset = Cresci18("train", root="./datasets", use_only_labelled=True, label_mapping=[5,0,"unlabelled"])
test_dataset = Cresci18("test", root="./datasets", use_only_labelled=True, label_mapping=[5,0,"unlabelled"])

save_processed_dataset(train_dataset, test_dataset, "Cresci18", dim_reducer, False, "./datasets/ProcessedDatasets/", device="cuda")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9854.34it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 20834 | 0

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9129.17it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 2593 | 5

In [4]:
# Dataset: Caverlee11
dim_reducer = joblib.load("./datasets/ProcessedDatasets/UmapModel.joblib")

train_dataset = Caverlee11("train", root="./datasets", label_mapping=[4,0])
test_dataset = Caverlee11("test", root="./datasets", label_mapping=[4,0])

save_processed_dataset(train_dataset, test_dataset, "Caverlee11", dim_reducer, False, "./datasets/ProcessedDatasets/", device="cuda")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8323.35it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 31943 | 0

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10505.98it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 3948 | 0

In [5]:
# Dataset: Twibot20
dim_reducer = joblib.load("./datasets/ProcessedDatasets/UmapModel.joblib")

train_dataset = Twibot20("train", root="./datasets", label_mapping=[6, 0])
test_dataset = Twibot20("test", root="./datasets", label_mapping=[6, 0])

save_processed_dataset(train_dataset, test_dataset, "Twibot20", dim_reducer, False, "./datasets/ProcessedDatasets/", device="cuda")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8149.34it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 9504 | 0

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10212.08it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 1148 | 6

In [3]:
# Dataset: Twibot22
dim_reducer = joblib.load("./datasets/ProcessedDatasets/UmapModel.joblib")

train_dataset = Twibot22Improved("train", root="./datasets", label_mapping=[7,0], sub_sample_size=40000)
test_dataset = Twibot22Improved("test", root="./datasets", label_mapping=[7,0])

save_processed_dataset(train_dataset, test_dataset, "Twibot22", dim_reducer, False, "./datasets/ProcessedDatasets/", device="cuda")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2640.80it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--> Loading labels...
--> Successfully cached 50000 users in memory.
--> Successfully cached 100000 users in memory.
--> Successfully cached 150000 users in memory.
--> Successfully cached 200000 users in memory.
--> Successfully cached 250000 users in memory.
--> Successfully cached 300000 users in memory.
--> Successfully cached 350000 users in memory.
--> Successfully cached 400000 users in memory.
--> Successfully cached 450000 users in memory.
--> Successfully cached 500000 users in memory.
--> Successfully cached 550000 users in memory.
--> Successfully cached 600000 users in memory.
--> Successfully cached 650000 users in memory.
--> Successfully cached 700000 users in memory.
--> Successfully cached 750000 users in memory.
--> Successfully cached 800000 users in memory.
--> Successfully cached 850000 users in memory.
--> Successfully cached 900000 users in memory.
--> Successfully cached 950000 users in memory.
--> Successfully cached 1000000 users in memory.
Iteration: 39999 |

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2583.89it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--> Loading labels...
--> Successfully cached 50000 users in memory.
--> Successfully cached 100000 users in memory.
--> Successfully cached 150000 users in memory.
--> Successfully cached 200000 users in memory.
--> Successfully cached 250000 users in memory.
--> Successfully cached 300000 users in memory.
--> Successfully cached 350000 users in memory.
--> Successfully cached 400000 users in memory.
--> Successfully cached 450000 users in memory.
--> Successfully cached 500000 users in memory.
--> Successfully cached 550000 users in memory.
--> Successfully cached 600000 users in memory.
--> Successfully cached 650000 users in memory.
--> Successfully cached 700000 users in memory.
--> Successfully cached 750000 users in memory.
--> Successfully cached 800000 users in memory.
--> Successfully cached 850000 users in memory.
--> Successfully cached 900000 users in memory.
--> Successfully cached 950000 users in memory.
--> Successfully cached 1000000 users in memory.
Iteration: 99999 |

In [5]:
dataset = ProcessedDataset("train", "./datasets/ProcessedDatasets/", "Twibot20")
for i, sample in enumerate(dataset):
    print(f"Sample {i}: {sample[0]}")

Sample 0: tensor([13.7099,  2.3910,  3.4064, 10.0860,  5.9729, -0.8659,  4.0459,  2.6087,
         4.9711,  4.2914,  8.5119,  4.3589,  5.2895,  5.3714,  3.2928,  5.2101,
         5.5385,  4.3770,  5.5661,  4.5495,  4.8917,  4.8819,  2.9101,  5.7336,
         5.9148,  4.8779,  3.7627,  4.7805,  4.4024,  4.2711,  5.0184,  4.7344,
         5.2528,  6.1296,  5.0437,  5.2945,  4.8799,  4.6570,  3.5000,  3.7500,
         0.0000,  1.8615,  0.0000,  0.0000,  0.0000], dtype=torch.float64)
Sample 1: tensor([13.9462,  2.5370,  3.3947,  9.3989,  5.6866, -1.0203,  3.6546,  2.1581,
         5.2058,  4.6102,  8.4261,  4.4988,  5.4246,  5.3174,  2.9283,  5.0629,
         5.4440,  4.2906,  5.7204,  4.9566,  4.7020,  5.0492,  2.6922,  5.5110,
         6.0893,  4.8390,  3.8939,  4.7098,  4.4492,  4.3475,  5.1004,  4.7749,
         5.1402,  5.9670,  5.2544,  4.9577,  4.9586,  4.7800,  3.0000,  2.5000,
         0.0000,  3.5029,  0.0000,  0.0000,  0.0000], dtype=torch.float64)
Sample 2: tensor([13.6910,  2.

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



Sample 8763: tensor([14.0394,  2.6547,  3.4519,  9.4403,  5.5822, -0.5114,  3.4477,  2.0867,
         5.1170,  4.5754,  8.6178,  4.5551,  5.3589,  5.3120,  2.7665,  5.0124,
         5.6334,  4.1843,  5.9111,  5.0188,  4.7177,  5.0928,  2.4381,  5.4684,
         5.9217,  4.6474,  3.8753,  4.6470,  4.3335,  4.4725,  5.1533,  4.6741,
         5.1939,  6.1530,  5.3229,  4.8255,  5.0588,  4.6853,  2.3333,  3.0000,
         0.0000,  0.3984,  0.0000,  0.0000,  0.0000], dtype=torch.float64)
Sample 8764: tensor([13.7639,  2.4549,  3.3904, 10.0435,  5.9144, -0.7212,  3.8990,  2.5091,
         4.9823,  4.3407,  8.5969,  4.3862,  5.2909,  5.3695,  3.1675,  5.1987,
         5.5682,  4.3448,  5.6296,  4.6346,  4.8698,  4.9123,  2.7656,  5.6781,
         5.9071,  4.8285,  3.7784,  4.7362,  4.3935,  4.3117,  5.0625,  4.7336,
         5.2421,  6.1578,  5.0928,  5.1764,  4.9292,  4.6611,  3.0000,  3.0000,
         0.0000,  1.4152,  0.0000,  0.0000,  0.0000], dtype=torch.float64)
Sample 8765: tensor([13.